# scRareBench | High-Level scVI | Dataset 3 — Wu breast-cancer atlas

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This notebook intentionally keeps the workflow high-level:

`load_dataset(3) → MethodSpec(scVI) → benchmark_method() → HTML / multi-seed bundle`

Dataset-specific preparation is owned by **scRareBench**, not by this notebook.
The package revision is pinned to `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`, where Dataset 3 is pinned to the
audited immutable CELLxGENE H5AD and exposes the validated benchmark contract:

- 100,064 cells × 28,468 features
- label: `celltype_subset`
- integration/evaluation batch: `donor_id`
- canonical raw counts: `layers["counts"]`
- scIB-reference HVG policy: `global`
- registered rare-cell scenario metadata attached by the package

The only preprocessing below that belongs to the method is scVI's fixed global
Seurat-v3 4,000-HVG selection. Biological labels are never passed to scVI.

## 1. Install the pinned package revision and scVI

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
REPO = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-deps",
    REPO,
])

METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)

from scrarebench.runtime import setup_runtime

setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("scvi",),
    quiet=False,
)


## 2. Load Dataset 3 through the package

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset

adata = load_dataset(3)
info = dataset_info(adata)

# Short fail-fast contract check. No dataset preparation is performed here.
expected = {
    "dataset_key": "wu_breast_cancer_atlas",
    "label_key": "celltype_subset",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "global",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Unexpected Dataset 3 contract for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

if adata.shape != (100_064, 28_468):
    raise RuntimeError(
        f"Unexpected Dataset 3 shape: {adata.shape}; "
        "expected (100064, 28468)."
    )
if "counts" not in adata.layers:
    raise RuntimeError("Dataset 3 canonical raw-count layer 'counts' is missing.")

print("scRareBench:", scrarebench.__version__)
print(adata)
print("\nDataset contract:")
for key in (
    "dataset_key",
    "label_key",
    "batch_key",
    "count_layer",
    "scib_hvg_batch_mode",
    "benchmark_ready",
):
    print(f"  {key}: {info.get(key)}")


## 3. scVI configuration

In [ ]:
# Method seeds vary; the benchmark/evaluation seed stays fixed.
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a quick smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    # Fixed Dataset 3 method-side HVG policy from the completed audit.
    "n_hvg": 4000,
    "hvg_span": 0.3,

    # scVI architecture/training.
    "n_hidden": 128,
    "n_latent": 30,
    "n_layers": 2,
    "dropout_rate": 0.10,
    "dispersion": "gene-batch",
    "gene_likelihood": "nb",
    "max_epochs": 200,
    "batch_size": 256,
}


## 4. User-owned method adapter

This is the only method-specific section. `benchmark_method()` gives the runner
an isolated copy of the benchmark dataset for each method seed.

For Dataset 3, HVGs are deliberately selected **globally** from raw counts.
`donor_id` is still supplied to scVI as the batch covariate and remains the
batch used by downstream benchmark evaluation.


In [ ]:
import scanpy as sc
import scvi

from scrarebench import MethodOutput, dataset_info


def run_scvi(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("scib_hvg_batch_mode") != "global":
        raise RuntimeError(
            "Dataset 3 is expected to use the validated global HVG policy."
        )
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(
            f"Validated raw-count layer {count_layer!r} is missing."
        )

    # Method-owned preprocessing: fixed global Seurat-v3 HVGs.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    hvg_mask = (
        method_adata.var["highly_variable"]
        .fillna(False)
        .to_numpy(dtype=bool)
    )
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    if int(hvg_mask.sum()) != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} HVGs, selected {int(hvg_mask.sum())}."
        )

    method_adata = method_adata[:, hvg_mask].copy()

    # Labels are intentionally not used by the integration method.
    scvi.settings.seed = int(seed)
    scvi.model.SCVI.setup_anndata(
        method_adata,
        layer=count_layer,
        batch_key=batch_key,
    )

    model = scvi.model.SCVI(
        method_adata,
        n_hidden=int(config["n_hidden"]),
        n_latent=int(config["n_latent"]),
        n_layers=int(config["n_layers"]),
        dropout_rate=float(config["dropout_rate"]),
        dispersion=str(config["dispersion"]),
        gene_likelihood=str(config["gene_likelihood"]),
    )

    model.train(
        max_epochs=int(config["max_epochs"]),
        train_size=0.90,
        validation_size=0.10,
        batch_size=int(config["batch_size"]),
        early_stopping=True,
        early_stopping_patience=20,
        accelerator="auto",
        devices="auto",
    )

    latent = model.get_latent_representation()
    return MethodOutput(
        latent=latent,
        barcodes=method_adata.obs_names,
    )


## 5. Run the high-level benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scVI",
    runner=run_scvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir="/content/scrarebench_results/dataset3_scvi",
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        # HTML + bundle are sufficient for the multi-seed delivery.
        "write_pdf_report": False,
        "include_cell_ids": False,
    },
    # Dependencies were installed above.
    install_dependencies=False,
    finalize=True,
)


## 6. Inspect and download the final result

In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

if result.archive_path is not None:
    try:
        from google.colab import files
        files.download(str(result.archive_path))
    except ImportError:
        pass
